## Agentic RAG For A List Of Documents

In [1]:
import dotenv
%load_ext dotenv
%dotenv

In [2]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
papers = ["./datasets/lora_paper.pdf", "./datasets/longlora_efficient_fine_tuning.pdf"]

### Create the list of tools for documents

In [4]:
from summary_tool import create_summary_tool
from vector_search_tool import create_vector_search_tool
from pathlib import Path

In [5]:
paper_to_tools_dict = {}

for paper in papers:
    print(f"Creating {paper} tool")
    path = Path(paper)
    summary_tool = await create_summary_tool(document_fp=path, doc_name=path.stem)
    vector_tool = await create_vector_search_tool(document_fp=path, doc_name=path.stem)
    paper_to_tools_dict[path.stem] = [summary_tool, vector_tool]

Creating ./datasets/lora_paper.pdf tool
Creating ./datasets/longlora_efficient_fine_tuning.pdf tool


In [6]:
paper_to_tools_dict

{'lora_paper': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x227edbbc980>,
 'longlora_efficient_fine_tuning': [<llama_index.core.tools.query_engine.QueryEngineTool at 0x227ee5a87d0>,
  <llama_index.core.tools.query_engine.QueryEngineTool at 0x227ee67fce0>]}

In [8]:
all_tools = [t for paper in papers for t in paper_to_tools_dict[Path(paper).stem]]
all_tools

### Build the Tool Retriever for most relevant tools

In [20]:
from llama_index.core import VectorStoreIndex
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(all_tools, index_cls=VectorStoreIndex)

In [24]:
obj_retriever = obj_index.as_retriever(similarity_top_k=2)

In [30]:
retrieved_tools = obj_retriever.retrieve("Explain to me what is the Lora and why it's being used.")

for tool in retrieved_tools:
    print(tool.metadata.name)

lora_paper_vector_query_engine_tool
lora_paper_summary_query_engine_tool


In [31]:
retrieved_tools = obj_retriever.retrieve("Explain to me what is LongLoRA and why it's being used.")

for tool in retrieved_tools:
    print(tool.metadata.name)

longlora_efficient_fine_tuning_summary_query_engine_tool
longlora_efficient_fine_tuning_vector_query_engine_tool


In [32]:
retrieved_tools = obj_retriever.retrieve("Compare LongLoRA with Lora.")

for tool in retrieved_tools:
    print(tool.metadata.name)

longlora_efficient_fine_tuning_summary_query_engine_tool
longlora_efficient_fine_tuning_vector_query_engine_tool


### Build the Agent

In [33]:
from llama_index.llms.anthropic import Anthropic


llm = Anthropic(model="claude-haiku-4-5")

In [34]:
from llama_index.core.agent import FunctionAgent

agent = FunctionAgent(
    system_prompt=""" \
                You are an AI agent programmed to respond to questions based on a 
                list of documents. Always utilize the tools available 
                to generate answers, ensuring that responses are based directly on the 
                provided materials rather than on any pre-existing knowledge. 
                All your responses should be formatted in markdown text.
                """,
    tool_retriever=obj_retriever, 
    llm=llm, 
    verbose=True)

### Test the Agent with user queries

In [35]:
response = await agent.run(user_msg="Explain to me what is the Lora and why it's being used.")

print(response)

[tick] add: AgentWorkflowStartEvent(user_msg="Explain to me what is the Lora and why it's being ...", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="Explain to me what is the Lora and why it's being used.")])], current...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='                 You are an AI agent programmed to respond to que...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'usage': {'input_tokens': 7

In [36]:
response = await agent.run(user_msg="Explain to me what is LongLoRA and why it's being used.")

print(response)

[tick] add: AgentWorkflowStartEvent(user_msg="Explain to me what is LongLoRA and why it's being ...", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="Explain to me what is LongLoRA and why it's being used.")])], current...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='                 You are an AI agent programmed to respond to que...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'usage': {'input_tokens': 7

In [37]:
response = await agent.run(user_msg="Compare LongLoRA with Lora.")

print(response)

[tick] add: AgentWorkflowStartEvent(user_msg='Compare LongLoRA with Lora.', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Compare LongLoRA with Lora.')])], current_agent_name='Agent')
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='                 You are an AI agent programmed to respond to que...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'usage': {'input_tokens': 759, 'output_tokens': 214}, 'stop_reas

In [38]:
print(str(response))

## Comparison: LongLoRA vs LoRA

Based on the document, here's a comprehensive comparison:

### **Core Differences**

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning method that adds trainable low-rank decomposition matrices to model weights. It significantly reduces the number of trainable parameters compared to full model fine-tuning and works effectively for standard context lengths.

**LongLoRA** builds upon LoRA principles but extends them specifically for long-context scenarios. It combines LoRA's parameter efficiency with techniques optimized for handling extended sequence lengths in large language models.

### **Key Distinctions**

| Aspect | LoRA | LongLoRA |
|--------|------|----------|
| **Primary Focus** | Standard-length context fine-tuning | Long-context fine-tuning |
| **Efficiency** | Parameter-efficient | Parameter-efficient + long-context optimized |
| **Context Capability** | Effective for typical sequence lengths | Enables extended context window